# 03 — Training

Multi-task MobileNetV4 (timm). Checkpoints every 500 steps to survive Colab disconnects.
Threshold tuning + calibration run after training completes.

In [ ]:
import os, pathlib
if not pathlib.Path('zahava-tzniut').exists():
    !git clone https://github.com/zahava-networks/zahava-tzniut.git
os.chdir('zahava-tzniut')
!pip install -q -r requirements.txt timm torch torchvision
assert pathlib.Path('.env').exists(), 'upload .env first'

In [ ]:
# Pull labels + human review from HF
from huggingface_hub import hf_hub_download
from pipelines.common import require_env
for f in ('labels.parquet', 'human_review.parquet'):
    try:
        hf_hub_download(repo_id=require_env('HF_DATASET_REPO'), filename=f, repo_type='dataset', local_dir='manifests')
    except Exception as e:
        print(f'skip {f}: {e}')

In [ ]:
# Mount Drive for checkpoint backup (so disconnections don't lose them)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/tzniut/checkpoints
!ln -sfn /content/drive/MyDrive/tzniut/checkpoints models/checkpoints

In [ ]:
# Train (resumable — re-run if Colab disconnects)
from pipelines.training.train import train
best = train()
print('best checkpoint:', best)

In [ ]:
# Tune per-attribute thresholds on the validation set
from pipelines.training.threshold_tuner import tune
tune()

In [ ]:
# Calibrate (temperature scaling)
from pipelines.training.calibrator import calibrate
calibrate()